# Slash — Detect Suspicious Value Transfers in Poker
## Pair Ranking and Evidence Retrieval

This notebook provides a clean, reproducible starting point for the competition. It:

- discovers the mounted competition data automatically;
- summarizes the public labels and chronological split;
- trains a fast model on interpretable relational features;
- retrieves schema-valid evaluation evidence using chip-flow signals;
- validates and writes `submission.csv`.

> **Important:** unlisted development pairs are unknown, not negative. This baseline demonstrates the data contract and should be improved with richer temporal and action-level features.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:.4f}".format)

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working")
REQUIRED_FILES = {
    "players.parquet", "hands.parquet", "seats.parquet", "actions.parquet",
    "development_labels.csv", "development_evidence.csv",
    "evaluation_pairs.csv", "sample_submission.csv",
}

candidate_dirs = {
    path.parent
    for path in INPUT_ROOT.rglob("players.parquet")
    if REQUIRED_FILES.issubset(child.name for child in path.parent.iterdir())
}
if len(candidate_dirs) != 1:
    raise FileNotFoundError(
        "Add the competition data from Kaggle's Input panel and rerun. "
        f"Found {len(candidate_dirs)} compatible directories."
    )
DATA_DIR = candidate_dirs.pop()
print(f"Competition data: {DATA_DIR}")

## 1. Load and inspect the public data

Only the columns needed by this baseline are loaded. The large action table remains available for richer decision-level features.

In [ ]:
players = pd.read_parquet(DATA_DIR / "players.parquet")
hands = pd.read_parquet(
    DATA_DIR / "hands.parquet",
    columns=["hand_id", "phase", "big_blind", "started_at"],
)
labels = pd.read_csv(DATA_DIR / "development_labels.csv")
development_evidence = pd.read_csv(DATA_DIR / "development_evidence.csv")
evaluation_pairs = pd.read_csv(DATA_DIR / "evaluation_pairs.csv")

summary = pd.DataFrame({"Rows": [
    len(players), len(hands),
    int(hands["phase"].eq("development").sum()),
    int(hands["phase"].eq("evaluation").sum()),
    len(labels), int(labels["label"].sum()),
    len(development_evidence), len(evaluation_pairs),
]}, index=[
    "Players", "Hands", "Development hands", "Evaluation hands",
    "Public labelled pairs", "Public positive pairs",
    "Public evidence rows", "Evaluation pairs",
])
summary.style.format({"Rows": "{:,}"})

## 2. Understand the labels

`development_labels.csv` contains confirmed targets and hard negatives. Pairs absent from this file are **unknown**, not negative.

The disclosed target families are directed transfer, soft play, and coordinated isolation. `other_coordination` is reserved for evaluation.

In [ ]:
label_counts = (
    labels.groupby(["label_status", "behavior_family"])
    .size().rename("pairs").reset_index()
)
display(label_counts)

positive_counts = labels.loc[labels["label"].eq(1), "behavior_family"].value_counts().sort_index()
ax = positive_counts.plot.bar(
    color=["#7755CC", "#E05A47", "#2A9D8F"], figsize=(8, 4),
    title="Public positive pairs by behavior family",
)
ax.set_xlabel("")
ax.set_ylabel("Labelled pairs")
ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

## 3. Engineer features and train a fast model

We aggregate shared hands, chip flow, contribution imbalance, folds, and showdowns for every labelled and evaluation pair. These interpretable features deliberately ignore action order and cards, leaving substantial room for stronger solutions.

In [ ]:
from collections import defaultdict
from itertools import combinations

FEATURE_NAMES = [
    "log_shared_hands", "mean_gross_transfer_bb", "directional_imbalance",
    "absolute_directional_flow_bb", "both_showdown_rate", "one_fold_rate",
    "mean_contribution_gap_bb", "max_single_hand_transfer_bb",
]


def pair_key(left, right):
    return tuple(sorted((left, right)))


def feature_vector(values):
    shared, ab, ba, both_showdown, one_fold, gap, max_transfer = values
    gross = ab + ba
    return np.asarray([
        np.log1p(shared), gross / max(shared, 1.0),
        max(ab, ba) / max(gross, 1e-9), abs(ab - ba) / max(shared, 1.0),
        both_showdown / max(shared, 1.0), one_fold / max(shared, 1.0),
        gap / max(shared, 1.0), max_transfer,
    ], dtype=float)


def seat_hands(path):
    current_hand, current_rows = None, []
    columns = ["hand_id", "player_id", "total_contribution", "net_chips", "folded", "went_to_showdown"]
    for batch in pq.ParquetFile(path).iter_batches(columns=columns, batch_size=250_000):
        for row in batch.to_pylist():
            if current_hand is not None and row["hand_id"] != current_hand:
                yield current_rows
                current_rows = []
            current_hand = row["hand_id"]
            current_rows.append(row)
    if current_rows:
        yield current_rows


development_map = {pair_key(r.player_1, r.player_2): r.pair_id for r in labels.itertuples(index=False)}
evaluation_map = {pair_key(r.player_1, r.player_2): r.pair_id for r in evaluation_pairs.itertuples(index=False)}
required_pairs = {"development": set(development_map), "evaluation": set(evaluation_map)}
phase_by_hand = dict(zip(hands["hand_id"], hands["phase"]))
big_blind_by_hand = dict(zip(hands["hand_id"], hands["big_blind"]))
aggregates = defaultdict(lambda: np.zeros(7, dtype=float))

for hand_number, rows in enumerate(seat_hands(DATA_DIR / "seats.parquet"), start=1):
    hand_id = rows[0]["hand_id"]
    phase = phase_by_hand[hand_id]
    big_blind = float(big_blind_by_hand[hand_id])
    by_player = {row["player_id"]: row for row in rows}
    for left, right in combinations(sorted(by_player), 2):
        pair = (left, right)
        if pair not in required_pairs[phase]:
            continue
        a, b = by_player[left], by_player[right]
        net_a, net_b = float(a["net_chips"]) / big_blind, float(b["net_chips"]) / big_blind
        transfer_ab = min(max(-net_a, 0.0), max(net_b, 0.0))
        transfer_ba = min(max(-net_b, 0.0), max(net_a, 0.0))
        values = aggregates[(phase, pair)]
        values[0] += 1
        values[1] += transfer_ab
        values[2] += transfer_ba
        values[3] += float(a["went_to_showdown"] and b["went_to_showdown"])
        values[4] += float(a["folded"] != b["folded"])
        values[5] += abs(float(a["total_contribution"]) - float(b["total_contribution"])) / big_blind
        values[6] = max(values[6], transfer_ab, transfer_ba)
    if hand_number % 500_000 == 0:
        print(f"Processed {hand_number:,} hands")

train_pairs = [pair_key(r.player_1, r.player_2) for r in labels.itertuples(index=False)]
evaluation_pair_keys = [pair_key(r.player_1, r.player_2) for r in evaluation_pairs.itertuples(index=False)]
train_x = np.vstack([feature_vector(aggregates[("development", p)]) for p in train_pairs])
evaluation_x = np.vstack([feature_vector(aggregates[("evaluation", p)]) for p in evaluation_pair_keys])
train_y = labels["label"].to_numpy(dtype=int)

pair_model = make_pipeline(
    SimpleImputer(strategy="median"), StandardScaler(),
    LogisticRegression(class_weight="balanced", max_iter=2_000, random_state=42),
)
pair_model.fit(train_x, train_y)
risk = pair_model.predict_proba(evaluation_x)[:, 1]

positive_mask = train_y == 1
behavior_model = make_pipeline(
    SimpleImputer(strategy="median"), StandardScaler(),
    LogisticRegression(class_weight="balanced", max_iter=2_000, random_state=42),
)
behavior_model.fit(train_x[positive_mask], labels.loc[positive_mask, "behavior_family"])
behavior = np.where(risk >= 0.5, behavior_model.predict(evaluation_x), "none")

submission = pd.read_csv(DATA_DIR / "sample_submission.csv")
submission["risk_score"] = submission["pair_id"].map(pd.Series(risk, index=evaluation_pairs["pair_id"]))
submission["predicted_behavior"] = submission["pair_id"].map(pd.Series(behavior, index=evaluation_pairs["pair_id"]))

coefficients = pd.DataFrame({"feature": FEATURE_NAMES, "coefficient": pair_model[-1].coef_[0]})
display(coefficients.sort_values("coefficient", key=np.abs, ascending=False))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
evaluation_pairs["shared_hands"].plot.hist(bins=40, ax=axes[0], color="#7755CC", title="Shared evaluation hands")
submission["risk_score"].plot.hist(bins=40, ax=axes[1], color="#2A9D8F", title="Model risk scores")
axes[0].set_xlabel("Hands per pair")
axes[1].set_xlabel("Risk score")
plt.tight_layout()
plt.show()
submission[["pair_id", "risk_score", "predicted_behavior"]].head()

## 4. Retrieve simple evaluation evidence

For the 100 highest-risk pairs, shared evaluation hands are ranked using chip flow and contribution imbalance. A strong entry should additionally use action order, cards, pot odds, opponent context, and repeated behavior.

In [ ]:
top_pairs = evaluation_pairs.assign(risk_score=risk).nlargest(100, "risk_score")
top_players = set(top_pairs["player_1"]) | set(top_pairs["player_2"])
evaluation_hand_ids = set(hands.loc[hands["phase"].eq("evaluation"), "hand_id"])
started_at_by_hand = dict(zip(hands["hand_id"], pd.to_datetime(hands["started_at"], utc=True)))

seat_batches = []
for batch in pq.ParquetFile(DATA_DIR / "seats.parquet").iter_batches(
    columns=["hand_id", "player_id", "total_contribution", "net_chips"], batch_size=250_000,
):
    frame = batch.to_pandas()
    selected = frame[
        frame["player_id"].isin(top_players)
        & frame["hand_id"].isin(evaluation_hand_ids)
    ]
    if not selected.empty:
        seat_batches.append(selected)

top_seats = pd.concat(seat_batches, ignore_index=True)
assert not top_seats.duplicated(["hand_id", "player_id"]).any()
hands_by_player = {p: set(g["hand_id"]) for p, g in top_seats.groupby("player_id")}
seat_lookup = top_seats.set_index(["hand_id", "player_id"])

evidence_rows = []
for pair in top_pairs.itertuples(index=False):
    shared = hands_by_player.get(pair.player_1, set()) & hands_by_player.get(pair.player_2, set())
    scored = []
    for hand_id in shared:
        a = seat_lookup.loc[(hand_id, pair.player_1)]
        b = seat_lookup.loc[(hand_id, pair.player_2)]
        big_blind = float(big_blind_by_hand[hand_id])
        net_a, net_b = float(a.net_chips) / big_blind, float(b.net_chips) / big_blind
        transfer = max(
            min(max(-net_a, 0), max(net_b, 0)),
            min(max(-net_b, 0), max(net_a, 0)),
        )
        contribution_gap = abs(float(a.total_contribution) - float(b.total_contribution)) / big_blind
        scored.append((transfer + 0.25 * contribution_gap, started_at_by_hand[hand_id], hand_id))
    for rank, (score, _, hand_id) in enumerate(sorted(scored, key=lambda x: (-x[0], x[1]))[:5], start=1):
        evidence_rows.append({
            "pair_id": pair.pair_id, "evidence_rank": rank,
            "hand_id": hand_id, "evidence_score": score,
        })

evidence = pd.DataFrame(evidence_rows)
evidence.head(10)

## 5. Create and validate `submission.csv`

Unused evidence positions must contain `NO_EVIDENCE`. Empty cells are rejected by Kaggle.

In [ ]:
for rank in range(1, 6):
    ranked = evidence.loc[evidence["evidence_rank"].eq(rank)].set_index("pair_id")["hand_id"]
    submission[f"evidence_hand_{rank}"] = submission["pair_id"].map(ranked).fillna("NO_EVIDENCE")

expected_columns = [
    "pair_id", "risk_score", "predicted_behavior",
    *[f"evidence_hand_{rank}" for rank in range(1, 6)],
]
assert list(submission.columns) == expected_columns
assert len(submission) == len(evaluation_pairs)
assert submission["pair_id"].is_unique
assert set(submission["pair_id"]) == set(evaluation_pairs["pair_id"])
assert submission["risk_score"].between(0, 1).all()
assert set(submission["predicted_behavior"]).issubset({
    "none", "directed_transfer", "soft_play",
    "coordinated_isolation", "other_coordination",
})
assert not submission.isna().any().any()

for row in submission[[f"evidence_hand_{rank}" for rank in range(1, 6)]].itertuples(index=False, name=None):
    submitted = [hand_id for hand_id in row if hand_id != "NO_EVIDENCE"]
    assert len(submitted) == len(set(submitted))
    assert set(submitted).issubset(evaluation_hand_ids)

submission_path = OUTPUT_DIR / "submission.csv"
submission.to_csv(submission_path, index=False)
print(f"Saved: {submission_path}")
print(f"Rows: {len(submission):,}")
print(f"Size: {submission_path.stat().st_size / 1_000_000:.1f} MB")
print("All submission checks passed.")
submission.head()

## 6. Improve and submit

Ideas for a stronger solution:

1. use chronological development validation;
2. compare each player with their behavior against other opponents;
3. engineer partner-facing folds, calls, raises, and heads-up checks;
4. model temporal bursts and repeated interactions;
5. train an evidence ranker with `development_evidence.csv`;
6. report hard-negative performance and benign alternatives.

### Submit your first score

After the run completes, click **Save Version**, open the saved version's **Output**, select `submission.csv`, and click **Submit to Competition**.

### Responsible interpretation

This dataset is fully synthetic. A high score identifies benchmark patterns; it does not establish intent or guilt in real-world play. Evidence should support human review rather than automatic enforcement.